# Swingtrader backtesting pilot

This notebook tests the daily-bar backtester introduced on PR #128.
It contains a real-data run against the local bronze database using raw OHLC prices.

The real-data section uses a simple trailing 20-session momentum score only to
exercise the backtester.

It is **not** intended as a production strategy or model.
The score uses information available through each session close, and orders execute
at the following session open.

## 1. Environment setup

Open the notebook from somewhere inside the `swingtrader-app` repository. The next
cell locates the repository root and adds `src` to `sys.path`, so an editable install
is not required for the imports.

In [ ]:
import os
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display


def find_repository_root(start: Path) -> Path:
    candidates = (start, *start.parents)
    for candidate in candidates:
        if (
            (candidate / "pyproject.toml").is_file()
            and (candidate / "src" / "swingtrader").is_dir()
        ):
            return candidate
    raise FileNotFoundError(
        "Could not locate the swingtrader-app repository. "
        "Start Jupyter from inside the repository."
    )


REPOSITORY_ROOT = find_repository_root(Path.cwd().resolve())
SOURCE_ROOT = REPOSITORY_ROOT / "src"

if str(SOURCE_ROOT) not in sys.path:
    sys.path.insert(0, str(SOURCE_ROOT))

print(f"Repository root: {REPOSITORY_ROOT}")

In [ ]:
from swingtrader.data.bronze.loaders import load_bronze_daily_prices
from swingtrader.core.paths import find_repo_root
from swingtrader.data.db import resolve_database_engine
from swingtrader.indicators import atr
from swingtrader.modeling.backtest import run_backtest

## 3. Real-data configuration

By default, `DATABASE_URL = None` uses the project's configured database, which is
normally `data/swingtrader.sqlite`.

To test all available tickers, set `MAX_TICKERS = None`. Starting with 30 tickers
makes the first run easier to inspect.

In [ ]:
# Database and universe
DATABASE_URL: str | None = None
PROVIDER = "yfinance"
TICKERS: list[str] | None = None
MAX_TICKERS: int | None = 30

# Backtest period
START_DATE = "2024-01-01"
END_DATE: str | None = None
WARMUP_CALENDAR_DAYS = 120

# Simple point-in-time test signal
SIGNAL_LOOKBACK = 20
MIN_SCORE = 0.05
ATR_LENGTH = 14

# Portfolio and execution
INITIAL_CASH = 100_000.0
RISK_FRACTION = 0.005
MAX_POSITIONS = 10
MAX_HOLDING_SESSIONS = 5
STOP_ATR_MULTIPLE = 1.0
REWARD_RISK_RATIO = 2.0
COMMISSION_RATE = 0.0025

# Optional artifact export
SAVE_RESULTS = False
OUTPUT_DIRECTORY = REPOSITORY_ROOT / "artifacts" / "backtest-pilot"

## 4. Load raw bronze OHLC data

The notebook deliberately requests only raw `open`, `high`, `low`, and `close`.
`adjusted_close` is not used in the simulation.

In [ ]:
analysis_start = pd.Timestamp(START_DATE)
load_start = (analysis_start - pd.Timedelta(days=WARMUP_CALENDAR_DAYS)).date()

if not DATABASE_URL and "SWINGTRADER_DATABASE_URL" not in os.environ:
    repo_root = find_repo_root()
    DATABASE_URL = f"sqlite+pysqlite:///{(repo_root / 'data' / 'swingtrader.sqlite').as_posix()}"

engine = resolve_database_engine(
    database_url=DATABASE_URL,
    initialize=False,
)

bronze_prices = load_bronze_daily_prices(
    engine=engine,
    provider=PROVIDER,
    tickers=TICKERS,
    start_date=load_start,
    end_date=END_DATE,
    columns=("open", "high", "low", "close"),
)

if bronze_prices.empty:
    raise ValueError(
        "No bronze rows were loaded. Check DATABASE_URL, provider, tickers, "
        "and the requested date range."
    )

available_tickers = sorted(bronze_prices["ticker"].dropna().unique())

if TICKERS is None:
    selected_tickers = (
        available_tickers
        if MAX_TICKERS is None
        else available_tickers[:MAX_TICKERS]
    )
else:
    selected_tickers = list(TICKERS)
    missing_tickers = sorted(set(selected_tickers) - set(available_tickers))
    if missing_tickers:
        print(f"Requested tickers without loaded rows: {missing_tickers}")

bronze_prices = bronze_prices.loc[
    bronze_prices["ticker"].isin(selected_tickers)
].copy()

prices = (
    bronze_prices
    .set_index(["provider", "ticker", "trading_date"])
    .sort_index()
    .loc[:, ["open", "high", "low", "close"]]
)

print(f"Loaded rows: {len(prices):,}")
print(f"Selected tickers: {len(selected_tickers):,}")
print(
    "Available dates: "
    f"{prices.index.get_level_values('trading_date').min().date()} to "
    f"{prices.index.get_level_values('trading_date').max().date()}"
)
display(prices.head())

## 5. Create a simple point-in-time signal frame

The score is the trailing close-to-close return over `SIGNAL_LOOKBACK` sessions.
Only rows at or above `MIN_SCORE` are submitted to the backtester. ATR is calculated
from the same raw OHLC frame and is known after the signal session closes.

In [ ]:
grouped_close = prices["close"].groupby(
    level=["provider", "ticker"],
    sort=False,
)
momentum_score = grouped_close.pct_change(periods=SIGNAL_LOOKBACK)

atr_values = atr(
    prices.loc[:, ["high", "low", "close"]],
    length=ATR_LENGTH,
)

signals = pd.DataFrame(
    {
        "score": momentum_score,
        "atr": atr_values,
    },
    index=prices.index,
)

signal_dates = signals.index.get_level_values("trading_date")
signal_mask = (
    (signal_dates >= pd.Timestamp(START_DATE))
    & np.isfinite(signals["score"])
    & np.isfinite(signals["atr"])
    & signals["atr"].gt(0)
    & signals["score"].ge(MIN_SCORE)
)

if END_DATE is not None:
    signal_mask &= signal_dates.le(pd.Timestamp(END_DATE))

signals = signals.loc[signal_mask].sort_index()

if signals.empty:
    raise ValueError(
        "No signals passed the filters. Lower MIN_SCORE, widen the date range, "
        "or select more tickers."
    )

print(f"Signal rows: {len(signals):,}")
print(
    "Signal dates: "
    f"{signals.index.get_level_values('trading_date').nunique():,}"
)
print(f"Tickers with signals: {signals.index.get_level_values('ticker').nunique():,}")
display(signals.head())

### Signal distribution

This is a diagnostic for the temporary momentum score. It helps determine whether
the threshold creates a reasonable number of candidates.

In [ ]:
display(signals["score"].describe().to_frame("score"))

ax = signals["score"].plot.hist(
    bins=1000,
    figsize=(10, 4),
    title="Cumulative submitted signal-score distribution",
    histtype="step",
    linewidth=2,
    cumulative=True,
    density=True,
)
ax.grid(zorder=1)
ax.set_yticks(np.arange(0, 1.01, 0.1))
ax.set_xlabel(f"{SIGNAL_LOOKBACK}-session trailing return")
plt.show()

## 6. Run the backtest

The price frame starts on the first submitted signal date. The simulator uses the
following observed session for entry and closes remaining positions at the final
available close.

In [ ]:
first_signal_date = signals.index.get_level_values("trading_date").min()
backtest_date_mask = prices.index.get_level_values("trading_date") >= first_signal_date
backtest_prices = prices.loc[backtest_date_mask]

backtest_result = run_backtest(
    backtest_prices,
    signals,
    initial_cash=INITIAL_CASH,
    risk_fraction=RISK_FRACTION,
    max_positions=MAX_POSITIONS,
    max_holding_sessions=MAX_HOLDING_SESSIONS,
    stop_atr_multiple=STOP_ATR_MULTIPLE,
    reward_risk_ratio=REWARD_RISK_RATIO,
    commission_rate=COMMISSION_RATE,
)

trades = backtest_result["trades"]
equity = backtest_result["equity"]
summary = backtest_result["summary"]

print(f"Completed trades: {len(trades):,}")
display(summary.to_frame("value"))

## 7. Inspect the portfolio path

In [ ]:
ax = equity["equity"].plot(
    figsize=(12, 5),
    title="Backtest portfolio equity",
    ylabel="Equity (SEK)",
)
ax.set_xlabel("Trading date")
ax.grid(True)
plt.show()

display(equity.tail(10))

## 8. Inspect transactions

In [ ]:
if trades.empty:
    print("No trades were completed.")
else:
    trade_columns = [
        "provider",
        "ticker",
        "signal_date",
        "entry_date",
        "exit_date",
        "score",
        "quantity",
        "entry_price",
        "exit_price",
        "stop_price",
        "take_profit_price",
        "reward_risk",
        "holding_sessions",
        "exit_reason",
        "ambiguous_intrabar",
    ]
    display(trades.loc[:, trade_columns].head(50))
    display(trades["exit_reason"].value_counts().to_frame("trades"))

## 9. Optional result export

Set `SAVE_RESULTS = True` in the configuration cell to write immutable result files
under `artifacts/backtest-pilot`.

In [ ]:
if SAVE_RESULTS:
    OUTPUT_DIRECTORY.mkdir(parents=True, exist_ok=True)
    run_stamp = pd.Timestamp.now().strftime("%Y%m%dT%H%M%S")

    trades_path = OUTPUT_DIRECTORY / f"trades_{run_stamp}.parquet"
    equity_path = OUTPUT_DIRECTORY / f"equity_{run_stamp}.parquet"
    summary_path = OUTPUT_DIRECTORY / f"summary_{run_stamp}.csv"

    trades.to_parquet(trades_path, index=False)
    equity.to_parquet(equity_path)
    summary.rename("value").to_csv(summary_path)

    print(f"Saved: {trades_path}")
    print(f"Saved: {equity_path}")
    print(f"Saved: {summary_path}")
else:
    print("SAVE_RESULTS is False; nothing was written.")

## Next experiment

After verifying that the simulator behaves as expected, replace the temporary
momentum score with an out-of-sample model prediction frame containing `score`.
Retain raw-price ATR in the `signals` frame and keep the same backtest call.